# Clustering — K-Means & DBSCAN

Unsupervised clustering on a personality-traits dataset: K-Means (with the elbow method and
silhouette score to choose *k*, plus distance-based outlier detection) and DBSCAN (with a
k-distance graph to choose `eps`), visualized in 2D via PCA.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

In [ ]:
!git clone https://github.com/armitakamari/ML-Fundamentals-Implementations.git
%cd ML-Fundamentals-Implementations

## Load & Prepare Data

In [ ]:
df = pd.read_csv("./data/personality_dataset.csv")
df = df.drop(columns=["Personality"])

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
df = df[numeric_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

## K-Means — Choosing *k* with the Elbow Method

In [ ]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure()
plt.plot(k_range, inertia, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (SSE)")
plt.title("Elbow Method")
plt.show()

**Choosing k:** `k_optimal` is set below based on the elbow plot above and confirmed with the
silhouette score in the next cell  

In [ ]:
k_optimal = 2
kmeans = KMeans(n_clusters=k_optimal, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

In [ ]:
sil_kmeans = silhouette_score(X_scaled, kmeans_labels)
print("Silhouette Score (KMeans):", sil_kmeans)
# Higher is better — compare this across different k_optimal values to confirm your choice.

## K-Means — Distance-Based Outlier Detection

In [ ]:
distances = np.min(kmeans.transform(X_scaled), axis=1)
threshold = np.percentile(distances, 95)
outliers_kmeans = distances > threshold

print("Number of KMeans outliers:", np.sum(outliers_kmeans))

## DBSCAN — Choosing eps with a k-Distance Graph

In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

distances = np.sort(distances[:, 4])

plt.figure()
plt.plot(distances)
plt.xlabel("Points")
plt.ylabel("5-NN Distance")
plt.title("k-distance Graph for DBSCAN")
plt.show()

In [ ]:
dbscan = DBSCAN(eps=0.8, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print("Number of clusters (DBSCAN):", n_clusters)
print("Number of noise points:", n_noise)

## Visualization (PCA to 2D)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("KMeans Clustering (PCA)")
plt.show()

In [ ]:
plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=dbscan_labels)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("DBSCAN Clustering (PCA)")
plt.show()